# VN30 Fundamentals Screener

Build a value-and-quality screener over the VN30 constituents using P/E, P/B, ROE, and debt-to-equity. We'll filter for cheap, profitable, low-leverage names and plot the survivors.

**Why this matters**: a one-line filter on quality + value is the single most-replicated equity result in academic finance. It also happens to be the bread-and-butter of every Vietnamese long-only fund.

## Setup

This notebook uses **mocked data** so it runs without an API key. Replace each `mock_*` block with the real DataCore call shown in the comment once your `DATACORE_API_KEY` is set.

In [ ]:
import pandas as pd  # noqa
import numpy as np  # noqa
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)

## Step 1 — Pull VN30 constituents

In production you'd run `dc.dataset('equity.vn30.constituents').to_pandas()`. Here we mock a representative slice of the real VN30 with sector tags.

In [ ]:
# Replace with: constituents = dc.dataset('equity.vn30.constituents').to_pandas()
constituents = pd.DataFrame([
    {'ticker': 'VCB', 'name': 'Vietcombank',         'sector': 'Banks'},
    {'ticker': 'BID', 'name': 'BIDV',                'sector': 'Banks'},
    {'ticker': 'CTG', 'name': 'VietinBank',          'sector': 'Banks'},
    {'ticker': 'TCB', 'name': 'Techcombank',         'sector': 'Banks'},
    {'ticker': 'MBB', 'name': 'MB Bank',             'sector': 'Banks'},
    {'ticker': 'VPB', 'name': 'VPBank',              'sector': 'Banks'},
    {'ticker': 'ACB', 'name': 'ACB',                 'sector': 'Banks'},
    {'ticker': 'VIC', 'name': 'Vingroup',            'sector': 'Real Estate'},
    {'ticker': 'VHM', 'name': 'Vinhomes',            'sector': 'Real Estate'},
    {'ticker': 'VRE', 'name': 'Vincom Retail',       'sector': 'Real Estate'},
    {'ticker': 'NVL', 'name': 'Novaland',            'sector': 'Real Estate'},
    {'ticker': 'KDH', 'name': 'Khang Dien House',    'sector': 'Real Estate'},
    {'ticker': 'VNM', 'name': 'Vinamilk',            'sector': 'Consumer Staples'},
    {'ticker': 'MSN', 'name': 'Masan Group',         'sector': 'Consumer Staples'},
    {'ticker': 'SAB', 'name': 'Sabeco',              'sector': 'Consumer Staples'},
    {'ticker': 'MWG', 'name': 'Mobile World',        'sector': 'Consumer Disc.'},
    {'ticker': 'PNJ', 'name': 'Phu Nhuan Jewelry',   'sector': 'Consumer Disc.'},
    {'ticker': 'FPT', 'name': 'FPT Corporation',     'sector': 'Technology'},
    {'ticker': 'HPG', 'name': 'Hoa Phat Group',      'sector': 'Materials'},
    {'ticker': 'HSG', 'name': 'Hoa Sen Group',       'sector': 'Materials'},
    {'ticker': 'GAS', 'name': 'PetroVietnam Gas',    'sector': 'Energy'},
    {'ticker': 'PLX', 'name': 'Petrolimex',          'sector': 'Energy'},
    {'ticker': 'POW', 'name': 'PV Power',            'sector': 'Utilities'},
    {'ticker': 'GVR', 'name': 'Vietnam Rubber',      'sector': 'Materials'},
    {'ticker': 'BVH', 'name': 'Bao Viet Holdings',   'sector': 'Insurance'},
    {'ticker': 'SSI', 'name': 'SSI Securities',      'sector': 'Financials'},
    {'ticker': 'STB', 'name': 'Sacombank',           'sector': 'Banks'},
    {'ticker': 'TPB', 'name': 'TPBank',              'sector': 'Banks'},
    {'ticker': 'HDB', 'name': 'HDBank',              'sector': 'Banks'},
    {'ticker': 'BCM', 'name': 'Becamex IDC',         'sector': 'Real Estate'},
])
print(f'{len(constituents)} VN30 constituents')
constituents.head()

## Step 2 — Pull latest fundamentals

Replace with `dc.dataset('equity.fundamentals.latest').to_pandas(tickers=constituents['ticker'].tolist())`.

In [ ]:
# Mocked latest fundamentals (realistic VN ranges as of mid-2025)
rng = np.random.default_rng(7)
n = len(constituents)
fundamentals = pd.DataFrame({
    'ticker':       constituents['ticker'],
    'price_vnd':    rng.integers(15_000, 130_000, n),
    'mcap_t_vnd':   rng.uniform(8, 480, n).round(1),       # trillion VND
    'pe':           rng.uniform(5, 28, n).round(1),
    'pb':           rng.uniform(0.7, 4.5, n).round(2),
    'roe_pct':      rng.uniform(6, 28, n).round(1),
    'de':           rng.uniform(0.1, 2.4, n).round(2),     # debt/equity
    'div_yield':    rng.uniform(0, 8, n).round(1),
})
df = constituents.merge(fundamentals, on='ticker')
df.head()

## Step 3 — Apply the screener

Classic value-plus-quality cuts:

- **P/E < 15** — not too expensive on earnings
- **ROE > 15%** — profitable use of equity
- **D/E < 1.0** — conservatively financed

In [ ]:
mask = (df['pe'] < 15) & (df['roe_pct'] > 15) & (df['de'] < 1.0)
screened = df[mask].sort_values('roe_pct', ascending=False)

print(f'{mask.sum()} of {len(df)} VN30 names pass the screen\n')
screened[['ticker', 'name', 'sector', 'pe', 'pb', 'roe_pct', 'de', 'div_yield']].head(5)

## Step 4 — Visualize the universe

Plot P/E vs ROE, colored by sector. The screener corner (low P/E, high ROE) is the bottom-right — highlight the survivors with edge markers.

In [ ]:
fig, ax = plt.subplots()
sectors = df['sector'].unique()
cmap = plt.get_cmap('tab10')

for i, sector in enumerate(sectors):
    sub = df[df['sector'] == sector]
    ax.scatter(sub['pe'], sub['roe_pct'], s=sub['mcap_t_vnd'] * 1.2,
               c=[cmap(i % 10)], alpha=0.7, label=sector,
               edgecolors='black', linewidths=0.5)

# Mark survivors
ax.scatter(screened['pe'], screened['roe_pct'], s=200,
           facecolors='none', edgecolors='red', linewidths=2,
           label='Passes screen')

for _, row in screened.iterrows():
    ax.annotate(row['ticker'], (row['pe'], row['roe_pct']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.axvline(15, color='grey', linestyle='--', alpha=0.5)
ax.axhline(15, color='grey', linestyle='--', alpha=0.5)
ax.set_xlabel('P/E')
ax.set_ylabel('ROE (%)')
ax.set_title('VN30: P/E vs ROE (bubble size = market cap in T VND)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Step 5 — Sector breakdown of survivors

Where is the screener finding opportunities? A quick bar chart tells you whether you're concentrated in one sector — important for risk-budgeting.

In [ ]:
sector_counts = screened['sector'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
sector_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('# names passing screen')
ax.set_title('Sector concentration of value-quality survivors')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

sector_counts

## Step 6 — Export for downstream use

Save the screened list to CSV so your trading system or other notebooks can pick it up.

In [ ]:
from pathlib import Path

out_dir = Path('../../data')
out_dir.mkdir(exist_ok=True)
out_path = out_dir / 'vn30_screen_latest.csv'
screened.to_csv(out_path, index=False)
print(f'Wrote {len(screened)} rows to {out_path}')

## Next steps

- `02-sector-rotation.ipynb` — rotate across sectors instead of picking names
- `03-factor-investing/01-fama-french-vn.ipynb` — formalize value + size as factors
- `01-getting-started/03-caching-and-performance.ipynb` — cache fundamentals across re-runs